# A. Introduction

SinoNom Annotation is an end-to-end pipeline for preparing and annotating Sino-Nôm inscription data. It extracts inscription content from PDF documents, detects character regions in page images, and provides a Gradio interface for verifying text, bounding boxes, character conditions, reading order, and crop areas. The resulting structured annotations can be exported as a ZIP file for downstream OCR, restoration, and digital humanities research.

For installation instructions, configuration, and technical details, see the [SinoNom Annotation](https://github.com/VongHaiYen11/SinoNom-Annotation)

# B. Pipeline Execution

## 1. Clone the Repository and Prepare models

Before running the annotation pipeline, clone the project repository and install the required dependencies.

This step will:

* clone the **SinoNom-Annotation** repository from GitHub;
* download or load the required text detection models and model weights;

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/VongHaiYen11/SinoNom-Annotation.git"
REPO_DIR = Path("/kaggle/working/SinoNom-Annotation")

os.chdir("/kaggle/working")

# Delete existing repo folder
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

# Clone fresh repo
print("Cloning repository...")
subprocess.run(
    ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
    check=True
)

print("Done:", REPO_DIR)

In [ ]:
from pathlib import Path
import shutil
import os

REPO_DIR = Path("/kaggle/working/SinoNom-Annotation")

# Direct paths to checkpoint files
DAMAGE_CONFIG_SOURCE = Path(
    "/kaggle/input/models/nhutminhdang/autohdr-detection-models/other/default/2/damage_detect.py"
)

DAMAGE_MODEL_SOURCE = Path(
    "/kaggle/input/models/nhutminhdang/autohdr-detection-models/other/default/2/damage_detect.pth"
)

# Target checkpoint directory in repository
CKPT_TARGET_DIR = REPO_DIR / "text_detection/models/ckpts"

# Dist
DIST_SOURCE_DIR = Path(
    "/kaggle/input/models/nhutminhdang/autohdr-detection-models/other/default/2/dist"
)

DIST_TARGET_DIR = REPO_DIR / "text_detection/models/dists"


# Validate source files
if not DAMAGE_CONFIG_SOURCE.is_file():
    raise FileNotFoundError(
        f"Cannot find damage detection config: {DAMAGE_CONFIG_SOURCE}"
    )

if not DAMAGE_MODEL_SOURCE.is_file():
    raise FileNotFoundError(
        f"Cannot find damage detection model: {DAMAGE_MODEL_SOURCE}"
    )

if not DIST_SOURCE_DIR.is_dir():
    raise FileNotFoundError(
        f"Cannot find dist folder: {DIST_SOURCE_DIR}"
    )


# Create target directories
CKPT_TARGET_DIR.mkdir(parents=True, exist_ok=True)
DIST_TARGET_DIR.mkdir(parents=True, exist_ok=True)


# Copy damage detection files
shutil.copy2(
    DAMAGE_CONFIG_SOURCE,
    CKPT_TARGET_DIR / "damage_detect.py"
)

shutil.copy2(
    DAMAGE_MODEL_SOURCE,
    CKPT_TARGET_DIR / "damage_detect.pth"
)

print(f"Copied: {DAMAGE_CONFIG_SOURCE}")
print(f"Copied: {DAMAGE_MODEL_SOURCE}")


# Copy dist folder
shutil.copytree(
    DIST_SOURCE_DIR,
    DIST_TARGET_DIR,
    dirs_exist_ok=True,
)


# Make OCR executable executable
ocr_executable = DIST_TARGET_DIR / "det_model" / "det_model"

if not ocr_executable.is_file():
    raise FileNotFoundError(
        f"Cannot find OCR executable: {ocr_executable}"
    )

os.chmod(
    ocr_executable,
    os.stat(ocr_executable).st_mode | 0o111
)


print(f"Checkpoint folder ready: {CKPT_TARGET_DIR}")
print(f"Dist folder ready: {DIST_TARGET_DIR}")
print(f"OCR executable ready: {ocr_executable}")

## 2. Upload the PDF and its image folder

Before running the remaining cells, add a dataset/ folder containing:

- the source `.json` file;
- a **flat folder** of page/inscription images used by the annotation app.

The image filename without its extension must match the `ky_hieu` produced by text extraction. After attaching the dataset, edit `SOURCE_JSON`, and `IMAGE_DIR` in the next cell. 


In [ ]:
from pathlib import Path

REPO_DIR = Path('/kaggle/working/SinoNom-Annotation')

IMAGE_DIR = Path('/kaggle/input/datasets/ynvnghi/sample-dataset/images') 
SOURCE_JSON = Path('/kaggle/input/datasets/ynvnghi/sample-dataset/student_030.json')

IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png'}
if not IMAGE_DIR.is_dir():
    raise FileNotFoundError(f'Image folder not found: {IMAGE_DIR}')

images = sorted(path for path in IMAGE_DIR.iterdir()
                if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES)
if not images:
    raise ValueError(f'No supported images found directly inside: {IMAGE_DIR}')
if len({path.stem for path in images}) != len(images):
    raise ValueError('Image filenames must have unique stems.')

print('Extracted JSON:', SOURCE_JSON)
print('Image folder:', IMAGE_DIR)
print('Images:', len(images))

In [ ]:
import json
from pathlib import Path

records = json.loads(SOURCE_JSON.read_text(encoding='utf-8'))
if not isinstance(records, list):
    raise ValueError(f'Source JSON must contain a JSON array: {SOURCE_JSON}')

print('Source JSON:', SOURCE_JSON)
print('Records:', len(records))
print('Images:', len(images))

## 3. Install all repository requirements

This installs directly into the current Kaggle Python environment. Kaggle may print dependency-conflict warnings for preinstalled packages that this repository does not use; those warnings are intentionally ignored in this notebook.


In [ ]:
import subprocess
import sys

REPO_PYTHON = sys.executable
subprocess.run(
    [REPO_PYTHON, '-m', 'pip', 'install', '--upgrade',
     '-r', str(REPO_DIR / 'requirements.txt')],
    check=True,
)
print('Repository requirements installed into:', REPO_PYTHON)

## 4. Launch the Gradio annotation app

The app receives the uploaded image folder and the JSON produced by the previous cell. Per-image saves and Save-all data are written under `/kaggle/working/annotations`. Stop this cell to shut down Gradio.

For student_114 backups, run the cell from `kaggle_upload/restore_in_kaggle.ipynb` immediately before launching. Keep SOURCE_JSON pointing to the original `student_114.json` in Input. APP_LAUNCHER restores missing saved Content sections in memory without editing the source.


In [ ]:
import os
import subprocess
from pathlib import Path

ANNOTATION_DIR = Path('/kaggle/working/annotations')
ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)

command = [
    str(REPO_PYTHON), '-u', str(globals().get('APP_LAUNCHER', REPO_DIR / 'gradio/app.py')),
    '--image-dir', str(IMAGE_DIR.resolve()),
    '--source-json', str(SOURCE_JSON.resolve()),
    '--output-dir', str(ANNOTATION_DIR.resolve()),
    '--server-name', '0.0.0.0',
    '--port', '7830',
    '--share',
    '--show-detection-logs',
]

environment = dict(os.environ)
environment['MPLBACKEND'] = 'Agg'
environment['PYTHONUNBUFFERED'] = '1'
environment['SINONOM_REPO_DIR'] = str(REPO_DIR.resolve())

subprocess.run(
    command,
    cwd=REPO_DIR,
    env=environment,
    check=True,
)